# Работа 3. Подготовка сигналов подшипников к анализу

Источник записей — `data/Bearing.mat`. Проценты в четвёртом задании характеризуют долю дефектного сигнала в искусственной смеси. Они не являются измерениями реального износа.

## Подготовка

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import loadmat
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler

DATA = Path("data")


def check(task, value):
    try:
        if task == 1:
            ok = (isinstance(value, dict) and all(k in value for k in
                  ["normal", "roller", "inner", "outer"])
                  and value["inner"].size == 120_000
                  and round(np.abs(value["inner"]).sum()) == 25057)
        elif task == 2:
            ok = (len(value) == 4 and all(np.asarray(v).shape == (120_000,)
                  for v in value) and round(sum(np.abs(v).sum() for v in value)) == 94201)
        elif task == 3:
            ok = (len(value) == 4 and all(np.asarray(v).shape == (117, 4000)
                  for v in value) and np.array_equal(value[0][0, 1000:], value[0][1, :3000]))
        elif task == 4:
            X, y = value
            ok = (np.asarray(X).shape == (37, 1) and np.asarray(y).shape == (37, 3)
                  and np.allclose(y[0], 0) and np.isclose(y.sum(), 1950)
                  and np.isclose(X[0, 0], 0.0726744486205))
        elif task == 5:
            arr = np.asarray(value, dtype=float)
            ok = arr.shape == (14,) and np.isfinite(arr).all() and round(arr.sum()) == 45545
        elif task == 6:
            arr = np.asarray(value, dtype=float)
            ok = (arr.shape == (37, 1) and np.allclose(arr.mean(axis=0), 0, atol=1e-12)
                  and np.allclose(arr.std(axis=0, ddof=1), 1, atol=1e-12))
        elif task == 7:
            ok = (len(value) == 3 and all(isinstance(v, RandomForestRegressor) for v in value)
                  and [v.n_estimators for v in value] == [400, 400, 600]
                  and [v.max_depth for v in value] == [35, 35, 50])
        else:
            raise ValueError("неизвестный номер")
    except Exception as exc:
        print(f"❌ Задание {task}: {type(exc).__name__}: {exc}")
        return
    print(f"{'✅' if ok else '❌'} Задание {task}" +
          (" выполнено" if ok else ": проверьте результат"))


print("Папка с данными найдена:", DATA.exists())
print("Файлы:", [p.name for p in DATA.iterdir()] if DATA.exists() else [])

Папка с данными найдена: True
Файлы: ['Bearing.mat']


## 1. Чтение данных

Функция `loadmat` загружает MAT-файл в словарь `bearing`. Для дальнейших вычислений используются записи `normal`, `roller`, `inner` и `outer`.

In [2]:
bearing = loadmat(DATA / "Bearing.mat")
print([key for key in bearing if not key.startswith("__")])

['normal', 'inner', 'roller', 'outer']


In [3]:
check(1, bearing)

✅ Задание 1 выполнено


## 2. Подготовка одномерных массивов

Метод `.ravel()` преобразует каждую запись в вектор из 120 000 отсчётов. Четыре состояния подшипника сохраняются в отдельных переменных.

In [4]:
normal = bearing["normal"].ravel()
roller = bearing["roller"].ravel()
inner = bearing["inner"].ravel()
outer = bearing["outer"].ravel()
print("Длины:", [len(v) for v in [normal, roller, inner, outer]])

Длины: [120000, 120000, 120000, 120000]


In [5]:
check(2, (normal, roller, inner, outer))

✅ Задание 2 выполнено


## 3. Выделение окон

Функция `segment_signal` формирует фрагменты длиной 4000 отсчётов с перекрытием 75%. Шаг равен 1000 отсчётов, поэтому соседние окна содержат 3000 общих значений. Для каждой записи получаются 117 полных окон.

![Почему окна одного сигнала сильно похожи](visualizations/02_утечка_и_проверка.png)

In [6]:
def segment_signal(signal, length=4000, overlap=0.75):
    step = int(length * (1 - overlap))
    windows = [signal[start:start + length]
               for start in range(0, len(signal) - length + 1, step)]
    return np.stack(windows)

segments_normal = segment_signal(normal)
segments_roller = segment_signal(roller)
segments_inner = segment_signal(inner)
segments_outer = segment_signal(outer)
print("Окон:", segments_normal.shape)
print("Общая часть соседних окон: последние 3000 отсчётов первого окна "
      "и первые 3000 отсчётов следующего (75%).")

Окон: (117, 4000)
Общая часть соседних окон: последние 3000 отсчётов первого окна и первые 3000 отсчётов следующего (75%).


In [7]:
check(3, (segments_normal, segments_roller, segments_inner, segments_outer))

✅ Задание 3 выполнено


## 4. Построение синтетической выборки

К нормальной записи добавляются по 12 смесей для каждого дефекта. Используется выражение `normal + mixing_coefficient * (defect - normal)`. Признаком служит RMS сигнала после удаления среднего.

В `X_matrix` собраны 37 примеров с одним признаком. В `y_matrix` три столбца: внутреннее кольцо, внешнее кольцо и ролик. Коэффициент смеси умножается на 100 и записывается в столбец соответствующего дефекта.

![Synthetic signal mixture (синтетическая смесь сигналов) и RMS](visualizations/03_synthetic_signal_mixture_and_RMS.png)

In [8]:
mixing_coefficients = [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 1.0]

def rms(signal):
    return np.sqrt(np.mean((signal - np.mean(signal)) ** 2))

X_values = [rms(normal)]
y_values = [[0.0, 0.0, 0.0]]
for target_col, defect in enumerate([inner, outer, roller]):
    for mixing_coefficient in mixing_coefficients:
        mixture = normal + mixing_coefficient * (defect - normal)
        X_values.append(rms(mixture))
        target = np.zeros(3)
        target[target_col] = 100 * mixing_coefficient
        y_values.append(target)

X_matrix = np.asarray(X_values, dtype=float).reshape(-1, 1)
y_matrix = np.asarray(y_values, dtype=float)
print("X:", X_matrix.shape, "y:", y_matrix.shape)

X: (37, 1) y: (37, 3)


In [9]:
check(4, (X_matrix, y_matrix))

✅ Задание 4 выполнено


## 5. Характеристики первого окна

Для первых 4000 отсчётов нормального сигнала вычисляются RMS, частота максимума спектра, энергии десяти полос по 500 Гц, доля пересечений нуля и отношение 75-го процентиля к 25-му для абсолютных амплитуд.

Среднее предварительно удаляется. При использовании `np.fft.rfft` последняя точка Найквиста исключается согласно условию. Полученные 14 значений объединяются в `features`.

In [10]:
N, fs = 4000, 10_000
x = normal[:N] - normal[:N].mean()
rms_value = np.sqrt(np.mean(x ** 2))
amplitude = np.abs(np.fft.rfft(x))[:-1]
frequency = np.fft.rfftfreq(N, d=1 / fs)[:-1]
dominant_frequency = frequency[np.argmax(amplitude)]
band_energy = np.array([
    np.sum(amplitude[(frequency >= low) & (frequency < low + 500)] ** 2)
    for low in range(0, 5000, 500)
])
zero_crossing_rate = np.count_nonzero(np.diff(np.sign(x))) / N
q25, q75 = np.quantile(np.abs(x), [0.25, 0.75])
features = np.array([
    rms_value, dominant_frequency, *band_energy,
    zero_crossing_rate, q75 / q25,
])
print("Число признаков:", len(features))
feature_names = ["RMS", "Доминирующая частота, Гц"] + [
    f"Энергия {low}–{low + 500} Гц" for low in range(0, 5000, 500)
] + ["Доля пересечений нуля", "q75 / q25"]
pd.Series(features, index=feature_names, name="Значение")

Число признаков: 14


RMS                             0.074005
Доминирующая частота, Гц     1727.500000
Энергия 0–500 Гц            10117.963704
Энергия 500–1000 Гц          1729.499158
Энергия 1000–1500 Гц         2550.037471
Энергия 1500–2000 Гц        26704.211339
Энергия 2000–2500 Гц          462.744173
Энергия 2500–3000 Гц          139.069672
Энергия 3000–3500 Гц          229.241530
Энергия 3500–4000 Гц         1812.923216
Энергия 4000–4500 Гц           35.390996
Энергия 4500–5000 Гц           32.659090
Доля пересечений нуля           0.324500
q75 / q25                       3.522740
Name: Значение, dtype: float64

In [11]:
check(5, features)

✅ Задание 5 выполнено


## 6. Стандартизация

Каждый столбец `X_matrix` центрируется и делится на выборочное стандартное отклонение с `ddof=1`. Если отклонение близко к нулю, вместо него берётся единица. Преобразованные данные сохраняются в `X_scaled`.

In [12]:
means = np.mean(X_matrix, axis=0)
stds = np.std(X_matrix, axis=0, ddof=1)
stds[stds < 1e-10] = 1.0
X_scaled = (X_matrix - means) / stds
print("Среднее:", np.mean(X_scaled, axis=0))
print("Стандартное отклонение (ddof=1):", np.std(X_scaled, axis=0, ddof=1))

Среднее: [1.57531645e-16]
Стандартное отклонение (ddof=1): [1.]


In [13]:
check(6, X_scaled)

✅ Задание 6 выполнено


## 7. Создание моделей

Для двух колец используются леса из 400 деревьев с максимальной глубиной 35. Для ролика задаются 600 деревьев и глубина 50. Во всех моделях `min_samples_split=2`, `min_samples_leaf=1`; значения `random_state` равны 42, 43 и 44.

Здесь создаются объекты `RandomForestRegressor`. Обучение не проводится, поскольку `.fit()` не вызывается.

In [14]:
model_inner = RandomForestRegressor(
    n_estimators=400, max_depth=35, min_samples_split=2,
    min_samples_leaf=1, random_state=42,
)
model_outer = RandomForestRegressor(
    n_estimators=400, max_depth=35, min_samples_split=2,
    min_samples_leaf=1, random_state=43,
)
model_roller = RandomForestRegressor(
    n_estimators=600, max_depth=50, min_samples_split=2,
    min_samples_leaf=1, random_state=44,
)
print("Деревьев:", [m.n_estimators for m in (model_inner, model_outer, model_roller)])

Деревьев: [400, 400, 600]


In [15]:
check(7, (model_inner, model_outer, model_roller))

✅ Задание 7 выполнено


## Вывод

Подготовлены сигналы четырёх состояний подшипника, выделены перекрывающиеся окна и сформирована искусственная выборка. Рассчитаны признаки одного окна, стандартизирован RMS и настроены три модели регрессии.

37 примеров с одним признаком показывают организацию матриц данных и целей. По ним нельзя судить о качестве диагностики оборудования: для этого нужны обучение и независимая проверка. Перекрывающиеся фрагменты одной записи нельзя считать независимыми наблюдениями.